In [5]:
#imports
import pandas as pd
import requests

from pathlib import Path

import clean_reed
import merge_jobs

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from logger import logger

from config import (
    REED_CSV,
    LATEST_REED,
    REQUEST_TIMEOUT,
    UPDATE_PAGES,
    REED_API_KEY,
)

In [6]:
#create a retry session
def create_session():

    retry_strategy = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
    )

    session = requests.Session()

    adapter = HTTPAdapter(max_retries=retry_strategy)

    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session

In [7]:
# test the above block
if __name__ == "__main__":

    session = create_session()

    print(session)

In [8]:
# Load all existing Reed job IDs from the historical dataset
def get_existing_ids():
    
    if not Path(REED_CSV).exists():
        logger.info("No historical Reed dataset found.")
        return set()

    df = pd.read_csv(REED_CSV)

    if "jobId" not in df.columns:
        logger.warning("jobId column not found in reed_jobs.csv")
        return set()

    existing_ids = set(df["jobId"].astype(str))

    logger.info(f"Loaded {len(existing_ids)} existing Reed IDs.")

    return existing_ids

In [9]:
# test the above block
if __name__ == "__main__":

    ids = get_existing_ids()

    print(f"Existing IDs: {len(ids)}")

    print(list(ids)[:5])

2026-07-28 14:33:46 | INFO | Loaded 12680 existing Reed IDs.


Existing IDs: 12680
['57113383', '57069804', '57114781', '57064956', '57035279']


In [10]:
# fetch page
BASE_URL = "https://www.reed.co.uk/api/1.0/search"

RESULTS_PER_PAGE = 100


def fetch_page(session, page):

    params = {
        "resultsToTake": RESULTS_PER_PAGE,
        "resultsToSkip": (page - 1) * RESULTS_PER_PAGE,
    }

    try:

        response = session.get(
    BASE_URL,
    params=params,
    auth=(REED_API_KEY, ""),   
    timeout=REQUEST_TIMEOUT,
)

        response.raise_for_status()

        data = response.json()

        jobs = data.get("results", [])

        logger.info(f"Downloaded page {page}: {len(jobs)} jobs")

        return jobs

    except requests.RequestException as error:

        logger.error(f"Page {page} failed: {error}")

        return []

In [11]:
#test it
if __name__ == "__main__":

    session = create_session()

    jobs = fetch_page(session, 1)

    print(f"Jobs downloaded: {len(jobs)}")

    if jobs:

        print(type(jobs[0]))

        print(jobs[0].keys())

2026-07-28 14:33:47 | INFO | Downloaded page 1: 100 jobs


Jobs downloaded: 100
<class 'dict'>
dict_keys(['jobId', 'employerId', 'employerName', 'employerProfileId', 'employerProfileName', 'jobTitle', 'locationName', 'minimumSalary', 'maximumSalary', 'currency', 'expirationDate', 'date', 'jobDescription', 'applications', 'jobUrl'])


In [12]:
# collect jobs
def collect_jobs(session):

    existing_ids = get_existing_ids()

    new_jobs = []

    logger.info(f"Existing Reed IDs: {len(existing_ids)}")

    for page in range(1, UPDATE_PAGES + 1):

        jobs = fetch_page(session, page)

        if not jobs:
            break

        page_new = 0

        for job in jobs:

            job_id = str(job["jobId"])

            if job_id not in existing_ids:

                new_jobs.append(job)

                existing_ids.add(job_id)

                page_new += 1

        logger.info(f"Page {page}: {page_new} new jobs")

        if page_new == 0:
            logger.info("No new jobs on this page. Stopping update.")
            break

    logger.info(f"Collected {len(new_jobs)} new Reed jobs.")

    return new_jobs

In [13]:
# testing
if __name__ == "__main__":

    session = create_session()

    jobs = collect_jobs(session)

    print(f"\nNew jobs downloaded: {len(jobs)}")

    if jobs:
        print(jobs[0])

2026-07-28 14:33:47 | INFO | Loaded 12680 existing Reed IDs.
2026-07-28 14:33:47 | INFO | Existing Reed IDs: 12680
2026-07-28 14:33:49 | INFO | Downloaded page 1: 100 jobs
2026-07-28 14:33:49 | INFO | Page 1: 89 new jobs
2026-07-28 14:33:49 | INFO | Downloaded page 2: 100 jobs
2026-07-28 14:33:49 | INFO | Page 2: 92 new jobs
2026-07-28 14:33:49 | INFO | Downloaded page 3: 100 jobs
2026-07-28 14:33:49 | INFO | Page 3: 79 new jobs
2026-07-28 14:33:49 | INFO | Downloaded page 4: 100 jobs
2026-07-28 14:33:49 | INFO | Page 4: 0 new jobs
2026-07-28 14:33:49 | INFO | No new jobs on this page. Stopping update.
2026-07-28 14:33:49 | INFO | Collected 260 new Reed jobs.



New jobs downloaded: 260
{'jobId': 56395557, 'employerId': 598528, 'employerName': 'IT Career Switch', 'employerProfileId': None, 'employerProfileName': None, 'jobTitle': 'Trainee Project Manager', 'locationName': 'DL11ND', 'minimumSalary': 25000.0, 'maximumSalary': 45000.0, 'currency': 'GBP', 'expirationDate': '27/08/2026', 'date': '29/01/2026', 'jobDescription': 'Project Manager Placement Programme - No Experience Required Are you looking to kick-start a new career in Project Management? We are recruiting for companies who are looking to employ our Project Management Traineeship graduates to keep up with their growth. The best part is you will not need any previous experience as full training will be provided. You will also have the reassurance of a job guarantee ( 25K- 45K) upon completion. We are curre... ', 'applications': 3, 'jobUrl': 'https://www.reed.co.uk/jobs/trainee-project-manager/56395557'}


In [14]:
# save the newly collected jobs
def save_jobs(jobs):

    if not jobs:
        logger.info("No new jobs to save.")
        return

    df = pd.DataFrame(jobs)

    df.to_csv(LATEST_REED, index=False)

    logger.info(f"Saved {len(df)} new jobs to {LATEST_REED}")

In [15]:
# update new raw dataset
from pathlib import Path

def update_raw_dataset():

    latest = pd.read_csv(LATEST_REED)

    if Path(REED_CSV).exists():

        historical = pd.read_csv(REED_CSV)

        combined = pd.concat(
            [historical, latest],
            ignore_index=True
        )

    else:

        combined = latest

    combined.drop_duplicates(
        subset="jobId",
        keep="first",
        inplace=True
    )

    combined.to_csv(REED_CSV, index=False)

    logger.info(
        f"Historical Reed dataset updated: {len(combined)} jobs."
    )

In [16]:
def main():

    logger.info("=" * 60)
    logger.info("Reed Updater Started")
    logger.info("=" * 60)

    session = create_session()

    jobs = collect_jobs(session)

    if jobs:
        save_jobs(jobs)
        update_raw_dataset()
    else:
        logger.info("No new Reed jobs found.")

    # Always clean and merge
    clean_reed.clean_reed()
    merge_jobs.merge_jobs()

    logger.info("=" * 60)
    logger.info("Reed Updater Completed")
    logger.info("=" * 60)


if __name__ == "__main__":
    main()

2026-07-28 14:33:49 | INFO | ============================================================
2026-07-28 14:33:49 | INFO | Reed Updater Started
2026-07-28 14:33:49 | INFO | ============================================================
2026-07-28 14:33:50 | INFO | Loaded 12680 existing Reed IDs.
2026-07-28 14:33:50 | INFO | Existing Reed IDs: 12680
2026-07-28 14:33:51 | INFO | Downloaded page 1: 100 jobs
2026-07-28 14:33:51 | INFO | Page 1: 89 new jobs
2026-07-28 14:33:51 | INFO | Downloaded page 2: 100 jobs
2026-07-28 14:33:51 | INFO | Page 2: 92 new jobs
2026-07-28 14:33:52 | INFO | Downloaded page 3: 100 jobs
2026-07-28 14:33:52 | INFO | Page 3: 79 new jobs
2026-07-28 14:33:52 | INFO | Downloaded page 4: 100 jobs
2026-07-28 14:33:52 | INFO | Page 4: 0 new jobs
2026-07-28 14:33:52 | INFO | No new jobs on this page. Stopping update.
2026-07-28 14:33:52 | INFO | Collected 260 new Reed jobs.
2026-07-28 14:33:52 | INFO | Saved 260 new jobs to C:\Users\Visakh H S\Labour_Market_Intelligence\data